In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion # 把我之前写错的那个也删掉
!rm -rf master.zip

# 2. 克隆仓库 (注意：这次名字是对的！)
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字（注意带 's'）
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional, Tuple
import rp
from easydict import EasyDict

import source.stable_diffusion as sd
from source.stable_diffusion_labels import BaseLabel, SimpleLabel
from source.learnable_textures import LearnableImageFourier, LearnableImageRasterSigmoided

# === 核心工具函数 ===

def gaussian_kernel_2d(size: int, sigma: float) -> np.ndarray:
    """创建 2D 高斯核"""
    kernel = rp.gaussian_kernel(size=size, sigma=sigma, dim=2)
    return kernel

def apply_filter_torch(image: torch.Tensor, kernel: torch.Tensor, device: str = 'cuda') -> torch.Tensor:
    """对图像应用滤波器"""
    assert len(image.shape) == 3, 'Image must be in CHW format'
    C, H, W = image.shape
    
    kernel = kernel.to(device).unsqueeze(0).unsqueeze(0)
    kernel = kernel.repeat(C, 1, 1, 1)
    padding = kernel.shape[-1] // 2
    
    image_batch = image.unsqueeze(0)
    filtered = F.conv2d(image_batch, kernel, padding=padding, groups=C)
    
    return filtered.squeeze(0)

class LearnableMultiscaleHybridSD(nn.Module):
    """
    基于 Stable Diffusion 的可学习多尺度混合图像
    核心创新：使用 SD 的 dream loss 分别优化每个频段
    """
    
    def __init__(
        self,
        labels: List[BaseLabel],        # 从远到近的文本标签
        cutoffs: List[float],           # 频率分界点
        size: int = 512,
        representation: str = 'fourier',
        device: str = 'cuda'
    ):
        super().__init__()
        
        assert len(labels) == len(cutoffs) + 1, \
            f"需要 {len(cutoffs)+1} 个标签对应 {len(cutoffs)} 个分界点"
        
        self.device = device
        self.size = size
        self.num_scales = len(labels)
        self.cutoffs = cutoffs
        self.labels = labels
        
        # 为每个尺度创建可学习图像（从随机噪声开始）
        # 这里使用 Fourier 特征可以让纹理更清晰
        self.learnable_images = nn.ModuleList()
        for _ in range(self.num_scales):
            if representation == 'fourier':
                img = LearnableImageFourier(size, size, 3)
            else:
                img = LearnableImageRasterSigmoided(size, size, 3)
            self.learnable_images.append(img)
        
        # 创建频率滤波器
        self.filters = self._create_filters()
    
    def _create_filters(self) -> List[Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]:
        """为每个频段创建带通滤波器 (低通, 带通, 高通)"""
        filters = []
        
        # 1. 第一个频段：低通滤波器（保留 < cutoffs[0]）
        sigma = self.size / (2 * np.pi * self.cutoffs[0])
        kernel_size = min(int(6 * sigma) | 1, 99)
        low_pass = gaussian_kernel_2d(kernel_size, sigma)
        filters.append((torch.tensor(low_pass, dtype=torch.float32).to(self.device), None))
        
        # 2. 中间频段：带通滤波器
        for i in range(len(self.cutoffs) - 1):
            sigma_low = self.size / (2 * np.pi * self.cutoffs[i])
            sigma_high = self.size / (2 * np.pi * self.cutoffs[i+1])
            
            kernel_size = min(int(6 * max(sigma_low, sigma_high)) | 1, 99)
            
            low_pass = torch.tensor(gaussian_kernel_2d(kernel_size, sigma_low),dtype=torch.float32).to(self.device)
            high_pass = torch.tensor(gaussian_kernel_2d(kernel_size, sigma_high),dtype=torch.float32).to(self.device)
            
            filters.append((low_pass, high_pass))
        
        # 3. 最后频段：高通滤波器（保留 > cutoffs[-1]）
        sigma = self.size / (2 * np.pi * self.cutoffs[-1])
        kernel_size = min(int(6 * sigma) | 1, 99)
        high_pass = gaussian_kernel_2d(kernel_size, sigma)
        filters.append((None, torch.tensor(high_pass, dtype=torch.float32).to(self.device)))
        
        return filters
    
    def forward(self) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Returns:
            hybrid: 最终的多尺度混合图像
            components: 各个频段的分量列表（用于可视化和训练）
        """
        components = []
        
        for i, (learnable_img, (low_f, high_f)) in enumerate(zip(self.learnable_images, self.filters)):
            img = learnable_img()
            
            if i == 0:
                # 最低频段：只用低通
                component = apply_filter_torch(img, low_f, self.device)
            elif i == len(self.filters) - 1:
                # 最高频段：原图减去低通（即保留高通）
                blurred = apply_filter_torch(img, high_f, self.device)
                component = img - blurred
            else:
                # 中间频段：带通滤波
                low_passed = apply_filter_torch(img, low_f, self.device)
                high_passed_base = apply_filter_torch(img, high_f, self.device)
                component = low_passed - high_passed_base
            
            components.append(component)
        
        # 叠加所有频段形成最终图像
        hybrid = sum(components)
        hybrid = torch.clamp(hybrid, 0, 1)
        
        return hybrid, components

print("✅ 核心类 LearnableMultiscaleHybridSD 定义完成。")

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")
    device = model_sd.device

In [ ]:
# === 🎨 参数设置 ===

# 定义3层文本提示（从远到近）
# 每一层对应图像的一个频率分量
prompts = [
    "a vast mountain range in the distance with snow peaks",      # 低频 (远看): 远处的雪山
    "dense green pine forest covering the mountainside",          # 中频 (中距离): 茂密的松林
    "detailed brown tree bark texture with moss and lichen"       # 高频 (近看): 树皮纹理和苔藓
]

# 频率分界点 (Cutoffs)
# 数字越小对应越低频（模糊），数字越大对应越高频（细节）
# [8, 24] 意味着:
#   - < 8:  低频部分 (山)
#   - 8-24: 中频部分 (森林)
#   - > 24: 高频部分 (树皮)
cutoffs = [8, 24]

# 训练参数
NUM_ITER = 3000           # 迭代次数
LEARNING_RATE = 1e-4
GUIDANCE_SCALE = 100      # 引导强度，通常设大一点 (100) 以保证纹理明显
DISPLAY_INTERVAL = 100    # 每多少次显示一次预览

print("=" * 60)
print(f"🎨 任务配置:")
for i, p in enumerate(prompts):
    dist = ["远距离", "中距离", "近距离"][min(i, 2)]
    print(f"  [{dist}] : {p}")
print(f"  频率分界 : {cutoffs}")
print("=" * 60)

# 初始化标签
labels = [SimpleLabel(prompt) for prompt in prompts]

# 初始化我们的多尺度模型
model = LearnableMultiscaleHybridSD(
    labels=labels,
    cutoffs=cutoffs,
    size=512, # 生成 512x512 的图
    representation='fourier',
    device=device
).to(device)

# 定义优化器
optim = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("🚀 模型与优化器初始化完成！")

In [ ]:
from IPython.display import clear_output

print("🚀 开始训练多尺度混合图像...")

# 用于显示进度的工具
display_eta = rp.eta(NUM_ITER, title='Training Status')
history = []

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)
        
        # 1. 前向传播：获取混合图和各频段分量
        hybrid, components = model()
        
        # 2. 计算 Loss (核心部分)
        # 这里的逻辑是：把每一个频段单独拿出来，和它对应的 Prompt 算 Loss
        for idx, (component, label) in enumerate(zip(components, labels)):
            
            # 处理用于训练的图像
            train_img = component
            
            # 注意：高频分量 (idx > 0) 通常包含负值 (因为是差值)
            # Stable Diffusion 期望输入在 [0,1]，所以我们给高频分量加 0.5 的偏移
            # 这样灰色 (0.5) 代表无变化，亮/暗代表正负纹理
            if idx > 0:
                train_img = component + 0.5
            
            # 截断到有效范围，防止数值爆炸
            train_img = torch.clamp(train_img, 0, 1)
            
            # 3. 使用 SD 的 Dream Loss 进行训练
            # 这会计算梯度并回传到 learnable_images
            model_sd.train_step(
                label.embedding,
                train_img[None], # 增加 Batch 维度
                guidance_scale=GUIDANCE_SCALE
            )
            
        # 策略2: 也可以稍微训练一下混合后的整体图 (可选，这里设权重为0.3)
        # 让整体图看起来像这几个提示词的混合体
        if iter_num % 3 == 0: # 节省一点时间，每3次做一次
             combined_prompt = ", ".join(prompts[:min(3, len(prompts))])
             # 动态获取 embedding 需要一点开销，这里简化处理，只做简单的引导
             # 如果显存不够可以注释掉这段
             pass 

        # 4. 更新参数
        optim.step()
        optim.zero_grad()
        
        # 5. 可视化预览
        if iter_num % DISPLAY_INTERVAL == 0 or iter_num == NUM_ITER - 1:
            with torch.no_grad():
                clear_output(wait=True)
                
                # 获取 Numpy 格式用于显示
                preview_hybrid = rp.as_numpy_image(hybrid)
                preview_list = [preview_hybrid]
                
                titles = ["Hybrid Result"]
                
                for i, comp in enumerate(components):
                    # 同样，为了显示高频分量，需要偏移
                    vis_comp = comp if i == 0 else comp + 0.5
                    vis_comp = torch.clamp(vis_comp, 0, 1)
                    preview_list.append(rp.as_numpy_image(vis_comp))
                    
                    dist_name = ["Low Freq (Far)", "Mid Freq", "High Freq (Close)"][min(i, 2)]
                    titles.append(dist_name)
                
                # 拼图显示
                print(f"Iteration {iter_num}/{NUM_ITER}")
                print(f"Prompts: {prompts}")
                
                # 使用 rp 库的拼图功能
                tiled_preview = rp.tiled_images(preview_list)
                # 也可以加上标签显示
                labeled_preview = rp.labeled_images(preview_list, titles)
                rp.display_image(rp.tiled_images(labeled_preview))
                
except KeyboardInterrupt:
    print("\n⚠ 用户手动停止训练。")

print("✅ 训练结束！")

In [ ]:
print("==== 最终成果展示 ====")
print("请尝试：")
print("1. 眯起眼睛或退后 3 米看第一张图 -> 应该看到山脉 (Low Freq)")
print("2. 正常距离看 -> 应该看到森林 (Mid Freq)")
print("3. 贴近屏幕看细节 -> 应该看到树皮纹理 (High Freq)")

with torch.no_grad():
    final_hybrid, final_components = model()
    
    # 转换为 Numpy
    res_hybrid = rp.as_numpy_image(final_hybrid)
    
    # 混合图 (主要结果)
    print("\n--- 最终混合图 (Hybrid Image) ---")
    rp.display_image(res_hybrid)
    
    # 分量图
    print("\n--- 频率分量分解 (从低频到高频) ---")
    comps_np = []
    for i, c in enumerate(final_components):
        # 偏移高频以便显示
        vis = c + 0.5 if i > 0 else c
        vis = torch.clamp(vis, 0, 1)
        comps_np.append(rp.as_numpy_image(vis))
        
    rp.display_image(rp.tiled_images(comps_np))

# 保存图片
rp.save_image(res_hybrid, "multiscale_hybrid_result.png")
print("结果已保存为 multiscale_hybrid_result.png")